In [1]:
import json
import jsonlines

from openai import OpenAI

In [ ]:
API_KEY = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

In [ ]:
results_dict = {}
for i in range(1, 5):
    with open(f"subset_{i}_student_solving.json", "r", encoding="utf-8") as f:
        results_dict[f"subset_{i}"] = json.load(f)


with open("aggregated_student_solving_grading_batch_results.json", "r", encoding="utf-8") as f:
    teacher_grading_results = json.load(f)

In [22]:
def extract_wrong_solving_question(grading_subset_results, student_outputs):
    results = []
    target_ids = set(grading_subset_results["wrong_ids"])
    
    for item in student_outputs:
        if item["question_id"] in target_ids:
            results.append({
                "question_id": item["question_id"],
                "question": item["question"],
                "target_answer": item["target_answer"]
            })
    return results

In [23]:
filtered_subset_1 = extract_wrong_solving_question(teacher_grading_results["subset_1"], results_dict['subset_1']) # dict, list 
filtered_subset_2 = extract_wrong_solving_question(teacher_grading_results["subset_2"], results_dict['subset_2'])
filtered_subset_3 = extract_wrong_solving_question(teacher_grading_results["subset_3"], results_dict['subset_3']) 
filtered_subset_4 = extract_wrong_solving_question(teacher_grading_results["subset_4"], results_dict['subset_4'])

In [6]:
wrong_ids_set_1 = set(teacher_grading_results["subset_1"]["wrong_ids"])
question_ids_set_1 = {item["question_id"] for item in filtered_subset_1}
print(wrong_ids_set_1 == question_ids_set_1)

wrong_ids_set_2 = set(teacher_grading_results["subset_2"]["wrong_ids"])
question_ids_set_2 = {item["question_id"] for item in filtered_subset_2}
print(wrong_ids_set_2 == question_ids_set_2)

wrong_ids_set_3 = set(teacher_grading_results["subset_3"]["wrong_ids"])
question_ids_set_3 = {item["question_id"] for item in filtered_subset_3}
print(wrong_ids_set_3 == question_ids_set_3)

wrong_ids_set_4 = set(teacher_grading_results["subset_4"]["wrong_ids"])
question_ids_set_4 = {item["question_id"] for item in filtered_subset_4}
print(wrong_ids_set_4 == question_ids_set_4)

True
True
True
True


In [26]:
results_dict["subset_1"][0]

{'question_id': 'q_1',
 'question': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?',
 'concepts': 'The problem involves basic arithmetic operations such as addition, multiplication, and division. It also requires understanding of fractions and percentages. The concept of finding a fraction of a number is involved. Therefore, this problem can be classified as a mathematical problem involving arithmetic operations, comparisons, ratios, and probabilities.',
 'model_solving': "Let's break down the problem step by step:\n\n### Step 1: Define Variables\n- Let \\( M \\) represent the number of seashells Mimi picked up.\n- Let \\( K \\) represent the number of seashells Kyle found.\n- Let \\( L \\) represent the number of seashells Leigh found.\n\n### Step 2: Express Given Information Using Variables\n- Mimi picked up 2 dozen seashells

In [7]:
INFO_USAGE_EXAMPLES = r"""
### Examples for (3) "info_usage" (DO NOT solve these examples)
- These examples are ONLY to demonstrate how to write "info_usage".
- In "info_usage", do NOT compute intermediate numeric results or give the final answer.
- Just describe (A) what information you will use, and (B) how you will use it (variables/equations/operations).

[Example 1]
question: "Sara has 30 stickers. She gives 12 stickers to her friend. How many stickers does she have left?"
info_usage:"I will use the initial number of stickers Sara has and the number of stickers she gives away. I will treat these as the starting quantity and the removed quantity, respectively. Using subtraction, I will subtract the given-away amount from the initial amount to determine the remaining number of stickers."

[Example 2]
question:"A box has 96 cookies. Each day for 7 days, Tom eats 8 cookies. At the end of the week, he packs the remaining cookies into bags that hold 5 cookies each. He makes as many full bags as possible and leaves the rest unpacked. How many cookies are left unpacked?"
info_usage:"I will use the initial number of cookies, the daily cookies eaten, and the number of days to determine total cookies eaten by multiplying the daily amount by the number of days. I will subtract that total from the initial amount to get the remaining cookies. Then I will use the bag capacity to determine how many full bags can be made by dividing the remaining cookies by the bag size and taking only whole bags. Finally, I will use the remainder from that division to identify how many cookies are left unpacked."
"""

In [8]:
def teacher_prompt(question_id, question, target_answer):
    prompt = f"""You are an AI expert with the world's best mathematical reasoning ability.
When solving complex problems, strictly adhere to the following "multi-step reasoning protocol".

**Problem:**
{question}

**Correct Answer:**
{target_answer}

**Your Task:**
You must produce FOUR things:

(1) Step-by-step solution (variable setup → relationships/equations → calculation → final answer)
- Define variables clearly.
- Build relationship equations from the question.
- Solve step-by-step with explicit algebra/arithmetic.
- If units exist, track units.
- Use the Chain-of-Thought approach to eliminate any logical leaps and describe each step in detail.
- If the result derived from your solution differs from the given correct answer, re-examine the entire solution process from the beginning and correct any errors.

(2) List ALL required items (math concepts/formulas/theorems/relationships/etc.) needed to solve THIS problem
- You must enumerate ALL required items.
- For EACH item, explain why it is needed.

(3) Describe the information you will use to solve the problem and how you will use it via the required items in (2)
- For EACH piece of information you will use to solve the problem, write a single combined description that includes:
  (A) what the information is (as you will use it for solving), and
  (B) how you will use it in the solution.
- While writing (3), if you realize that any required item is missing for solving the problem, you MUST revise (2) to include the missing item(s) first, and then write (3) based on the revised (2).
- Do NOT compute intermediate numeric results and Do NOT give the final answer. There are two examples: {INFO_USAGE_EXAMPLES}

(4) For EACH required item in (2), create ONE analogous practice problem (so N items → N practice problems)
- Each practice problem must be different from the original (different numbers/context).
- Each practice problem must primarily train that single concept.
- Provide a short step-by-step solution and final answer for each.
- Keep practice problems simpler than the original when possible.

**Response Format (JSON only):**
Return ONLY valid JSON. Use this exact schema:
{{
  "problem_id": "{question_id}",

  "step_by_step_solution": "Write a clear step-by-step solution as text. It must include: variable setup → relationship/equation setup → calculations.",

  "required_items": {{
    "items": [
      "A required math concept/formula/theorem/relationship/etc. (string)",
      "..."
    ],
    "why_needed": "Explain in detail why each item listed in required_items.items is necessary to solve this problem."
  }},

  "info_usage": "One combined text that states (A) what information you will use to solve the problem and (B) how you will use it (variable/equation/calculation).",

  "practice_problems": [
    {{
      "concept": "A string that must exactly match one element of required_items.items",
      "problem": "new analogous problem statement  (different from the original numbers/context)",
      "solution": "Step-by-step solution as text (variable setup → equations/relationships → calculations).",
      "final_answer": "Final answer(number) as string."
    }}
  ]
}}
"""
    return prompt

In [9]:
print(teacher_prompt(1, 2, 3))

You are an AI expert with the world's best mathematical reasoning ability.
When solving complex problems, strictly adhere to the following "multi-step reasoning protocol".

**Problem:**
2

**Correct Answer:**
3

**Your Task:**
You must produce FOUR things:

(1) Step-by-step solution (variable setup → relationships/equations → calculation → final answer)
- Define variables clearly.
- Build relationship equations from the question.
- Solve step-by-step with explicit algebra/arithmetic.
- If units exist, track units.
- Use the Chain-of-Thought approach to eliminate any logical leaps and describe each step in detail.
- If the result derived from your solution differs from the given correct answer, re-examine the entire solution process from the beginning and correct any errors.

(2) List ALL required items (math concepts/formulas/theorems/relationships/etc.) needed to solve THIS problem
- You must enumerate ALL required items.
- For EACH item, explain why it is needed.

(3) Describe the in

In [10]:
filtered_subset_4[0]

{'question_id': 'q_5605',
 'question': 'Susan has 3 fish tanks to fill. 1 fish tank contains 7 goldfish and 8 beta fish. The second fish tank contains twice as many fish as the first tank and the third fish tank has a third of the number of fish in the second fish tank. How many fish are in the third fish tank?',
 'target_answer': '10'}

In [ ]:
def convert_to_openai_batch_jsonl(data, output_file):
    # data(json) -> output_file(jsonl)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for example in data:
            
            analysis_prompt = teacher_prompt(
                question_id=example['question_id'],
                question=example['question'],
                target_answer=example['target_answer'],
                
            )
            
            # Create OpenAI Batch API request format
            batch_request = {
                "custom_id": example['question_id'],
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": "gpt-4o-mini-2024-07-18",  
                    "messages": [
                        {
                            "role": "system",
                            "content": "You are an AI expert with the world’s best mathematical reasoning ability."
                        },
                        {
                            "role": "user",
                            "content": analysis_prompt
                        }
                    ],
                    "response_format": {"type": "json_object"},
                    "temperature": 0.0,
                    "max_tokens": 3000
                }
            }
            
            f.write(json.dumps(batch_request) + '\n')

In [13]:
convert_to_openai_batch_jsonl(filtered_subset_1, "teacher의 student가 틀린 문제 풀이 결과 및 어떤 정보 어떻게/teacher_subset_batch_1.jsonl")
convert_to_openai_batch_jsonl(filtered_subset_2, "teacher의 student가 틀린 문제 풀이 결과 및 어떤 정보 어떻게/teacher_subset_batch_2.jsonl")
convert_to_openai_batch_jsonl(filtered_subset_3, "teacher의 student가 틀린 문제 풀이 결과 및 어떤 정보 어떻게/teacher_subset_batch_3.jsonl")
convert_to_openai_batch_jsonl(filtered_subset_4, "teacher의 student가 틀린 문제 풀이 결과 및 어떤 정보 어떻게/teacher_subset_batch_4.jsonl")

In [ ]:
subset_1_batch_file = client.files.create(
    file=open("teacher_subset_batch_1.jsonl", "rb"), 
    purpose="batch" 
)

In [15]:
subset_1_batch_file_id = subset_1_batch_file.id

In [ ]:
subset_1_batch = client.batches.create(
    input_file_id=subset_1_batch_file_id,
    endpoint="/v1/chat/completions", 
    completion_window="24h" 
)

In [ ]:
print(client.batches.retrieve(subset_1_batch.id))
print(client.batches.retrieve(subset_1_batch.id).status)

In [ ]:
content = client.files.content(client.batches.retrieve(subset_1_batch.id).output_file_id)
content.write_to_file("teacher_subset_batch_result_1.jsonl")

In [ ]:
subset_2_batch_file = client.files.create(
    file=open("teacher_subset_batch_2.jsonl", "rb"), 
    purpose="batch" 
)

subset_2_batch_file_id = subset_2_batch_file.id

In [ ]:
subset_2_batch = client.batches.create(
    input_file_id=subset_2_batch_file_id,
    endpoint="/v1/chat/completions", 
    completion_window="24h" 
)

In [ ]:
print(client.batches.retrieve(subset_2_batch.id))
print(client.batches.retrieve(subset_2_batch.id).status)

In [ ]:
content = client.files.content(client.batches.retrieve(subset_2_batch.id).output_file_id)
content.write_to_file("teacher_subset_batch_result_2.jsonl")

In [ ]:
subset_3_batch_file = client.files.create(
    file=open("teacher_subset_batch_3.jsonl", "rb"), 
    purpose="batch" 
)


subset_3_batch_file_id = subset_3_batch_file.id

In [ ]:
subset_3_batch = client.batches.create(
    input_file_id=subset_3_batch_file_id, 
    endpoint="/v1/chat/completions", 
    completion_window="24h" 
)

In [ ]:
print(client.batches.retrieve(subset_3_batch.id))
print(client.batches.retrieve(subset_3_batch.id).status)

In [ ]:
content = client.files.content(client.batches.retrieve(subset_3_batch.id).output_file_id)
content.write_to_file("teacher_subset_batch_result_3.jsonl")

In [ ]:
subset_4_batch_file = client.files.create(
    file=open("teacher_subset_batch_4.jsonl", "rb"), 
    purpose="batch" 
)

subset_4_batch_file_id = subset_4_batch_file.id

In [ ]:
subset_4_batch = client.batches.create(
    input_file_id=subset_4_batch_file_id, 
    endpoint="/v1/chat/completions", 
    completion_window="24h" 
)

In [ ]:
print(client.batches.retrieve(subset_4_batch.id))
print(client.batches.retrieve(subset_4_batch.id).status)

In [ ]:
content = client.files.content(client.batches.retrieve(subset_4_batch.id).output_file_id)
content.write_to_file("teacher_subset_batch_result_4.jsonl")

In [ ]:
import jsonlines

with jsonlines.open("teacher_subset_batch_result_1.jsonl") as f:
    sampled_batch_results = list(f)

print(sampled_batch_results[0].keys())

dict_keys(['id', 'custom_id', 'response', 'error'])


In [65]:
results_dict["subset_1"][3]

{'question_id': 'q_4',
 'question': "Emma's bank account has $100 in it. Each day of the week, she spends $8. At the end of the week, she goes to the bank and asks for as many $5 bills as her account can give her. She leaves the rest in the account. How many dollars remain in the account?",
 'concepts': 'The problem involves basic arithmetic operations such as addition, subtraction, multiplication, and division. It also requires understanding of fractions (since Emma is asking for $5 bills), which falls under the category of ratio and proportion. Additionally, it involves conditional statements (if-else) to determine how much money remains after spending each day.',
 'model_solving': "To solve this problem, we need to follow these steps:\n\n### Step 1: Define Variables\nLet \\( x \\) represent the amount of money Emma has at the beginning of the week ($100).\n\n### Step 2: Write Relations\nWe know that Emma spends $8 every day. Therefore, over the course of a week (7 days), she will sp

In [ ]:
print(sampled_batch_results[2]["response"]["body"]["choices"][0]["message"]["content"])


{
  "problem_id": "q_4",

  "step_by_step_solution": "Let E be the initial amount in Emma's bank account, which is $100. Let D be the daily spending amount, which is $8, and let W be the number of days in a week, which is 7. First, calculate the total amount spent in a week: Total spent = D * W = 8 * 7 = $56. Next, calculate the remaining amount in the account after one week: Remaining amount = E - Total spent = 100 - 56 = $44. Now, Emma goes to the bank to withdraw as many $5 bills as possible. To find out how many $5 bills she can withdraw, divide the remaining amount by 5: Number of $5 bills = Remaining amount / 5 = 44 / 5 = 8 (whole bills). The total amount withdrawn is 8 * 5 = $40. Finally, calculate the amount left in the account after the withdrawal: Amount left = Remaining amount - Total withdrawn = 44 - 40 = $4.",

  "required_items": {
    "items": [
      "Multiplication",
      "Subtraction",
      "Division",
      "Remainder calculation"
    ],
    "why_needed": "Multipli

In [68]:
print(sampled_batch_results[-1]["response"]["body"]["choices"][0]["message"]["content"])


{
  "problem_id": "q_1868",

  "step_by_step_solution": "Let C be the total number of cookies baked. Frank bakes 2 trays of cookies per day for 6 days, and each tray makes 12 cookies. Therefore, C = 2 trays/day * 12 cookies/tray * 6 days = 144 cookies. Frank eats 1 cookie each day for 6 days, so he eats a total of 6 cookies. The number of cookies after Frank eats is 144 - 6 = 138 cookies. Ted comes over on the sixth day and eats 4 cookies. Therefore, the number of cookies left after Ted eats is 138 - 4 = 134 cookies.",

  "required_items": {
    "items": [
      "Multiplication for total cookies baked",
      "Subtraction for cookies eaten by Frank",
      "Subtraction for cookies eaten by Ted"
    ],
    "why_needed": "Multiplication is needed to calculate the total number of cookies baked based on the number of trays and cookies per tray. Subtraction is needed to account for the cookies eaten by Frank and Ted, which reduces the total number of cookies available."
  },

  "info_usage"